# Analytics across multiple cases

1. Read all directories (projects) in a data directory
2. Upload each all text files in each project to the Pinecone index
3. Read Pinecone index
4. For a query, find relevant documents
5. Using Langchain, send the query and relevant documents to ChatGPT
6. Get the answer

In [1]:
import openai
import os
from langchain.document_loaders import DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import Pinecone as Pinecone_lch
from langchain.llms import OpenAI
from langchain.chains.question_answering import load_qa_chain
from casify_pinecone_lch import get_answer
from dotenv import load_dotenv, find_dotenv
import pinecone
# from casify_pinecone_lch import index_name
# from casify_pinecone_lch import MODEL
from langchain.document_loaders import TextLoader
from store_document import do_store_document

/home/mark/anaconda3/envs/CasifyAI/lib/python3.9/site-packages/pinecone/index.py:4: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
/home/mark/anaconda3/envs/CasifyAI/lib/python3.9/site-packages/langchain/llms/openai.py:202: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain.chat_models import ChatOpenAI`
  warnings.warn(
/home/mark/anaconda3/envs/CasifyAI/lib/python3.9/site-packages/langchain/llms/openai.py:789: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain.chat_models import ChatOpenAI`
  warnings.warn(


In [2]:
!ls /home/mark/projects/scaia/scaia-datasets/Police-reports | wc
reports_dir = '/home/mark/projects/scaia/scaia-datasets/Police-reports'
print(reports_dir)

    650    2594   16894
/home/mark/projects/scaia/scaia-datasets/Police-reports


In [3]:
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key  = os.getenv('OPENAI_API_KEY')

In [4]:
pinecone.init(
    api_key=os.getenv('PINECONE_API_KEY'),     
    environment=os.getenv('PINECONE_ENVIRONMENT')
)
index_name = os.getenv('PINECONE_INDEX_NAME')
print("index_name="+index_name)
pinecone.describe_index(index_name)
index_pc = pinecone.Index(index_name)
print(str(index_pc))
description = pinecone.describe_index(index_name)
print(description)

index_name=casify-test
IndexDescription(name='casify-test', metric='cosine', replicas=1, dimension=1536.0, shards=1, pods=1, pod_type='p1.x1', status={'ready': True, 'state': 'Ready'}, metadata_config=None, source_collection='')


In [5]:
def split_docs(documents, chunk_size=1000, chunk_overlap=20):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    splits = text_splitter.split_documents(documents)
    return splits

In [6]:
def load_docs(directory):
  loader = DirectoryLoader(directory)
  documents = loader.load()
  return documents

In [7]:
from pathlib import Path
# specify the path to the directory. Use Path('.') for current directory or specify a path like Path('/path/to/directory')
data_dir = "/home/mark/projects/scaia/scaia-datasets/Police-reports"
parent_directory = Path(data_dir)
# list all subdirectories
iter_count_for_analysis = 0
for item in parent_directory.iterdir():
    if item.is_dir():
        case_dir_name = item.name    
        #print("case_dir_name=" + case_dir_name)
        case_id = case_dir_name.split(' ')[0]        
        namespace = case_id
        #print("case_id=" + case_id)
        
        case_dir_path = Path(data_dir + "/" + case_dir_name)               
        #documents = load_docs(case_dir_path)
        #splits = split_docs(documents)        
        #embeddings = OpenAIEmbeddings(openai_api_key=openai.api_key)
        
        #index_pc.delete(deleteAll = True, namespace = namespace)
        #Pinecone_lch.from_documents(splits, embeddings, index_name=index_name, namespace=namespace)
        #pinecone.describe_index(index_name)
        #index_pc = pinecone.Index("casify")
        #description = pinecone.describe_index(index_name)
        #print(description)
        iter_count_for_analysis += 1
        #if (iter_count_for_analysis > 10):            
            #break
        #print("iter_count_for_analysis=" + str(iter_count_for_analysis))
        query = "Read over these police reports and identify whether an injury or death occurred in the police report. If you find either an injury or death, please explain what occurred."
        #print("Ask: " + query)
        answer=get_answer(query, namespace)     
        if (len(answer) > 80):
            print(str(iter_count_for_analysis) + ": " + case_id + " " + answer)        

14: 12H04588 The document does not provide any information about an injury or death occurring.
22: 14C07468 The police report does not specifically mention any injury or death. However, it mentions that a female, identified as Kassandra Jimenez, appeared to be passed out at a picnic table and paramedics were called for her. She was subsequently transported to Coastal Communities Hospital. The specific nature of her condition or health status isn't described in the report.
25: 12N02175 The police reports do not indicate that an injury or death occurred. The reports are primarily concerned with the suspect, Kerry Ann Privett, being charged for furnishing alcohol to a minor.
28: 12H11597 The police report does not mention any injury or death. It only discusses a charge against Matthew Bernard Cain for selling alcohol to a minor, which is a misdemeanor.
38: 12C08287 The police report does mention injuries, but does not provide specific details about who was injured or how the injuries occu

In [ ]:
#loader = TextLoader("../data/159/processed/pages/courtdoc_page_1.txt")
#document = loader.load()

In [ ]:
#documents = [document]

In [ ]:
#docs = split_docs(documents)

In [ ]:
#Pinecone_lch.from_documents(docs, embeddings, index_name=index_name, namespace=namespace)

In [ ]:
from langchain.document_loaders import DirectoryLoader

directory = pages_dir

def load_docs(directory):
  loader = DirectoryLoader(directory)
  documents = loader.load()
  return documents

documents = load_docs(directory)
len(documents)

https://python.langchain.com/en/latest/modules/indexes/text_splitters/getting_started.html

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

def split_docs(documents,chunk_size=1000,chunk_overlap=20):
  text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
  docs = text_splitter.split_documents(documents)
  return docs

docs = split_docs(documents)
print(len(docs))

### Print example

In [ ]:
print(docs[0].page_content)
print(len(docs))

In [ ]:
import openai
from langchain.embeddings.openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(openai_api_key=openai.api_key)

query_result = embeddings.embed_query("Hello world")
len(query_result)

https://python.langchain.com/en/latest/modules/indexes/vectorstores/examples/pinecone.html

In [ ]:
# write all data to index_lch
index_lch = Pinecone_lch.from_documents(docs, embeddings, index_name=index_name, namespace = namespace)

# if you already have an index, you can load it like this
index_lch = Pinecone_lch.from_existing_index(index_name, embeddings, namespace = namespace)
index_pc = pinecone.Index(index_name)
index_stats_response = index_pc.describe_index_stats()
print(index_stats_response)

In [ ]:
def get_similar_docs(query,num_sources=5,score=False):
  if score:
    similar_docs = index_lch.similarity_search_with_score(query,k=num_sources)
  else:
    similar_docs = index_lch.similarity_search(query,k=num_sources)
  return similar_docs


In [ ]:
from langchain.llms import OpenAI

llm = OpenAI(model_name=MODEL)

chain = load_qa_chain(llm, chain_type="stuff")

https://python.langchain.com/en/latest/use_cases/question_answering.html

In [ ]:
answer=get_answer("What is the charge?", "159")
print(answer)